# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

# Imports

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import anthropic
from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT

# Initialization

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")
    
MODEL_GPT = "gpt-4o-mini"
MODEL_CLAUDE = "claude-3-5-sonnet-latest"
openai = OpenAI()
claude = anthropic.Anthropic()

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


# System Message

In [3]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

# Tools

In [4]:
# Pricing tool

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499", "madrid": "$399"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [5]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [6]:
# Reserve ticket tool

def reserve_ticket(destination_city, name):
    print(f"Tool reserve_ticket called for {name} to {destination_city}")
    return f"Ticket reserved for {name} to {destination_city.title()}"

In [7]:
reserve_function = {
    "name": "reserve_ticket",
    "description": "Reserve a return ticket for a customer to a given city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to",
            },
            "name": {
                "type": "string",
                "description": "The name of the customer",
            }
        },
        "required": ["destination_city", "name"],
        "additionalProperties": False
    }
}

In [8]:
# Tool list
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": reserve_function}
]

In [9]:
# Method to handle the tools
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_ticket_price":
        city = arguments.get("destination_city")
        price = get_ticket_price(city)
        result = {"destination_city": city, "price": price}

    elif function_name == "reserve_ticket":
        city = arguments.get("destination_city")
        name = arguments.get("name")
        confirmation = reserve_ticket(city, name)
        result = {"destination_city": city, "name": name, "confirmation": confirmation}

    else:
        result = {"error": "Unknown tool called"}

    response = {
        "role": "tool",
        "content": json.dumps(result),
        "tool_call_id": tool_call.id
    }

    return response, arguments

# Talker Agent

In [10]:
import base64
from io import BytesIO
from PIL import Image
from IPython.display import Audio, display

from pydub import AudioSegment
from pydub.playback import play

def talker(message, voice="onyx"):
    response = openai.audio.speech.create(
      model="tts-1",
      voice=voice,
      input=message
    )
    
    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play(audio)

# Whisper Agent

In [11]:
def transcribe_audio(audio_file_path):
    try:
        # Debugging: Check if the file path is None or empty
        if audio_file_path is None or audio_file_path == "":
            return ""
        
        if not os.path.exists(audio_file_path):
            return f"Error: Audio file does not exist at path {audio_file_path}"

        # Check if file is empty
        if os.path.getsize(audio_file_path) == 0:
            return "Error: Audio file is empty"

        with open(audio_file_path, "rb") as audio_file:
            response = openai.audio.transcriptions.create(model="whisper-1", file=audio_file)        
            # Debugging: Check the response object structure
            print(f"OpenAI API response: {response.text}")
            # Extract the transcription text from the response
            return response.text
    except Exception as e:
        return f"An error occurred: {e}"

# Translator Agent

In [12]:
def translate(history, target_language="spanish"):
    dialogue_text = ""
    for turn in history:
        role = turn["role"].capitalize()
        if role in ["User", "Assistant"]:
            dialogue_text += f"{role}: {turn['content']}\n"

    if target_language.lower() == "english":
        return dialogue_text

    system_message = f"Translate the following English dialogue into {target_language.capitalize()}. "
    system_message += "Preserve the structure including 'User:' and 'Assistant:'. "
    system_message += "Return only the translation, not the original text."
    
    try:
        result = claude.messages.create(
            model=MODEL_CLAUDE,
            system=system_message,
            max_tokens=1024,
            messages=[{"role": "user", "content": dialogue_text}],
        )
        #return result.completion.strip()
        return result.content[0].text.strip()

    except Exception as e:
        print(f"Translation error: {str(e)}")
        return dialogue_text

In [13]:
def translate_streaming(history, target_language="spanish"):
    dialogue_text = ""
    for turn in history:
        role = turn["role"].capitalize()
        if role in ["User", "Assistant"]:
            dialogue_text += f"{role}: {turn['content']}\n"

    if target_language.lower() == "english":
        yield dialogue_text
        return

    system_message = f"Translate the following English dialogue into {target_language.capitalize()}. "
    system_message += "Preserve the structure including 'User:' and 'Assistant:'. And 'User' and 'Assistant' should be written in bold."
    system_message += "Return only the translation, not the original text."

    try:
        stream = claude.messages.stream(
            model=MODEL_CLAUDE,
            system=system_message,
            max_tokens=1024,
            messages=[{"role": "user", "content": dialogue_text}]
        )

        response = ""
        with stream as stream_response:
            for chunk in stream_response.text_stream:
                if chunk:
                    response += chunk
                    yield response

    except Exception as e:
        print(f"Streaming translation error: {str(e)}")
        yield dialogue_text

In [14]:
def chat(history, tone="formal"):
    
    system_message = "You are a helpful assistant for an Airline called FlightAI. "
    system_message += "Give short answers, no more than 1 sentence. "
    system_message += "Always be accurate. If you don't know the answer, say so."
    system_message += f"It is mandatory that you response using a {tone} tone."
    
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
        
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, args = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
        
    return response.choices[0].message.content

In [15]:
def respond_and_update(message, history, language, tone):

    if history is None:
        history = []
    
    # Add user message to history
    history.append({"role": "user", "content": message})

    # Get assistant's reply
    assistant_reply = chat(history, tone)

    # Update history with assistant's reply
    history.append({"role":"assistant", "content":assistant_reply})
    
    # Get translation
    translation = None
    for chunk in translate_streaming(history, language):
        translation = chunk

    #chat_pairs = [
    #    [m["content"], r["content"]]
    #    for m, r in zip(updated_history[::2], updated_history[1::2])
    #    if m["role"] == "user" and r["role"] == "assistant"
    #]

    return history, translation, "", assistant_reply

In [ ]:
def reset_mic_input():
    mic = gr.Audio(value=None, label="🎙️ Speak", type="filepath", interactive=True, sources=['microphone']) 
    return mic

# Function to clear the chatbot history and the mic input
def clear_all():
    return [], "", gr.State([]), "", gr.Audio(value=None, label="🎙️ Speak", type="filepath", interactive=True, sources=['microphone'])


# Interface
with gr.Blocks(css="""
    .translation-box {
        height: 500px;
        overflow-y: auto;
        border: 1px solid var(--block-border-color);
        border-radius: var(--block-radius);
        padding: 10px;
        background-color: var(--block-background-fill);
    }
    """) as ui:
    gr.Markdown("## 🛫 FlightAI Assistant\nChat in English → See translation")

    with gr.Row():
        chatbot_en = gr.Chatbot(label="✈️ English Chat (GPT)", height=500, type="messages")
        with gr.Column(elem_classes="translation-box"):
            gr.Markdown("### 🌍 Translation (Claude)")
            chatbot_tr = gr.Markdown(line_breaks=True)

    with gr.Row():
        user_input = gr.Textbox(label="📝 Write your message, the press Enter:", placeholder="Type your message here and press Enter.")

    # Add mic input
    with gr.Row():
        mic_input = gr.Audio(label="🎙️ Speak (Whisper-1)", type="filepath", interactive=True, sources=['microphone'])
        tone = gr.Dropdown(
            choices=[("😀 Formal", "formal"),
                     ("😎 Casual", "casual"),
                     ("😊 Friendly", "friendly"),
                     ("🥰 Sweet", "sweet"),
                     ("😏 Sarcastic", "sarcastic"),
                     ("😜 Snarky", "snarky"),
                     ("😤 Impatient", "impatient"),
                     ("😒 Condescending", "condescending"),
                     ("😡 Disrespectful", "disrespectful"),
                    ],  
            value="formal", label="🎭Select Tone:"
        )
        voice = gr.Dropdown(
            choices=[("👨‍🦰 Onyx (Male - Deep, calm)", "onyx"),
                     ("👨‍🦱 Echo (Male - Crisp, energetic)", "echo"),
                     ("👨‍🦲 Alloy (Male - Friendly, upbeat)", "alloy"),
                     ("👩‍🦰 Fable (Female - Warm, storytelling)", "fable"),
                     ("👩‍🦳 Nova (Female - Clear, bright)", "nova"),
                     ("👩‍🦱 Shimmer (Female - Smooth, soft)", "shimmer"),
                ],
            value="onyx",
            label="🎤 Assistant Voice"
        )
        language = gr.Dropdown(
            choices=[("Spanish", "Spanish"),
                     ("French", "French"),
                     ("German", "German"),
                     ("Italian", "Italian"),
                     ("Portuguese", "Portuguese"),
                     ("Japanese", "Japanese"),
                     ("Korean", "Korean"),
                     ("Chinese", "Chinese"),
                     ("Russian", "Russian"),
                     ("Turkish", "Turkish"),
                     ("Swedish", "Swedish"),
                     ("Norwegian", "Norwegian"),
                     ("Danish", "Danish"),
                     ("Finnish", "Finnish"),
                     ("Icelandic", "Icelandic"),
                     ("Czech", "Czech"),
                     ("Greek", "Greek"),
                     ("Romanian", "Romanian"),
                     ("Polish", "Polish"),
                     ("Dutch", "Dutch"),
                    ],
            value="Spanish", label="Select Language:"    
        )

    with gr.Row():
        clear = gr.Button("Clear")
    
    assistant_reply = gr.Textbox(visible=False)
        
    #history_state = gr.State([])

    # When user submits by typing
    user_input.submit(
        fn=respond_and_update,
        inputs=[user_input, chatbot_en, language, tone],
        outputs=[chatbot_en, chatbot_tr, user_input, assistant_reply]
    ).then(
        fn=talker,
        inputs=[assistant_reply, voice],
        outputs=None
    ).then(
        fn=reset_mic_input,
        inputs=None,
        outputs=mic_input
    )

    # When user speaks
    mic_input.change(
        fn=transcribe_audio,
        inputs=mic_input,
        outputs=user_input
    )

    # Clear button action
    clear.click(
        fn=clear_all,
        inputs=None,
        outputs=[chatbot_en, chatbot_tr, user_input, mic_input]
    )
    
ui.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\ssre_\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\ssre_\.cache\huggingface\gradio\frpc
